# Week 9 Optional Simple Prediction App

## Goal

This notebook creates the required Streamlit file `app.py`.

The Streamlit app will:

1. load the trained Week 7 XGBoost model with `joblib`
2. ask the user for Living Area, Beds, Baths, and Lot Size
3. prepare those inputs for the saved model
4. show the predicted home price
5. save the app in `results/09/app.py`

The Week 7 model uses more than four original features. For this simple Week 9 app, the four requested inputs are used directly. Other numeric features are left missing so the saved Week 7 preprocessing step can fill them with its learned median values. The school district is set to `Unknown`.


## 1. Import the packages

These packages are used to find files, load the saved model, and create the Streamlit file.

In [1]:
from pathlib import Path
import importlib.util
import subprocess
import sys

import joblib
import numpy as np
import pandas as pd

print("Packages are ready.")

Packages are ready.


## 2. Set the file paths

The trained Week 7 model is expected at:

`results/05/best_xgboost_model.joblib`

The Week 9 Streamlit app will be saved at:

`results/09/app.py`


In [2]:
working_dir = Path.cwd()

project_root = (
    working_dir.parent
    if working_dir.name.lower() == "notebooks"
    else working_dir
)

model_path = (
    project_root
    / "results"
    / "05"
    / "best_xgboost_model.joblib"
)

output_dir = project_root / "results" / "09"
output_dir.mkdir(parents=True, exist_ok=True)

app_path = output_dir / "app.py"

print("Model file:", model_path)
print("Week 9 output folder:", output_dir)

Model file: C:\Users\18327\IDX Exchange\results\05\best_xgboost_model.joblib
Week 9 output folder: C:\Users\18327\IDX Exchange\results\09


## 3. Check and load the Week 7 model

The Week 7 notebook saved the preprocessing steps and the trained XGBoost model in one joblib file.

In [3]:
if not model_path.exists():
    raise FileNotFoundError(
        f"Cannot find {model_path}. "
        "Please make sure best_xgboost_model.joblib is inside results/05/."
    )

model_bundle = joblib.load(model_path)

required_items = [
    "preprocessor",
    "model",
    "numeric_features",
    "categorical_features",
]

missing_items = [
    item
    for item in required_items
    if item not in model_bundle
]

if missing_items:
    raise ValueError(
        "The saved model is missing these items: "
        f"{missing_items}"
    )

preprocessor = model_bundle["preprocessor"]
model = model_bundle["model"]
numeric_features = model_bundle["numeric_features"]
categorical_features = model_bundle["categorical_features"]

print("Model loaded successfully.")
print()
print("Numeric features:")
print(numeric_features)
print()
print("Categorical features:")
print(categorical_features)

Model loaded successfully.

Numeric features:
['BedroomsTotal', 'BathroomsTotalInteger', 'YearBuilt', 'LivingArea', 'BuildingAreaTotal', 'LotSizeSquareFeet', 'LotSizeArea', 'Latitude', 'Longitude', 'ListPrice', 'OriginalListPrice', 'DaysOnMarket', 'GarageSpaces', 'Stories', 'bed_bath_ratio', 'property_age']

Categorical features:
['DistrictName']


## 4. Check the four Week 9 inputs

The assignment asks the app to use:

1. Living Area
2. Beds
3. Baths
4. Lot Size

The code below checks that the important matching columns are available in the saved Week 7 feature list.

In [4]:
all_features = numeric_features + categorical_features

week9_feature_check = pd.DataFrame(
    {
        "Week_9_Input": [
            "Living Area",
            "Beds",
            "Baths",
            "Lot Size",
        ],
        "Week_7_Column": [
            "LivingArea",
            "BedroomsTotal",
            "BathroomsTotalInteger",
            "LotSizeSquareFeet",
        ],
        "Found_in_Model": [
            "LivingArea" in all_features,
            "BedroomsTotal" in all_features,
            "BathroomsTotalInteger" in all_features,
            "LotSizeSquareFeet" in all_features,
        ],
    }
)

display(week9_feature_check)

,Week_9_Input,Week_7_Column,Found_in_Model
0,Living Area,LivingArea,True
1,Beds,BedroomsTotal,True
2,Baths,BathroomsTotalInteger,True
3,Lot Size,LotSizeSquareFeet,True


## 5. Create `app.py`

The following code writes the complete Streamlit application into `results/09/app.py`.

In [5]:
app_code = 'from pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport streamlit as st\n\n\n# -----------------------------\n# Page settings\n# -----------------------------\nst.set_page_config(\n    page_title="Home Price Prediction",\n    page_icon="🏠",\n    layout="centered",\n)\n\nst.title("Home Price Prediction")\nst.write(\n    "Enter the property information below and click the button "\n    "to get a predicted price."\n)\n\n\n# -----------------------------\n# Find and load the Week 7 model\n# -----------------------------\nAPP_DIR = Path(__file__).resolve().parent\nPROJECT_ROOT = APP_DIR.parent.parent\n\nMODEL_PATH = (\n    PROJECT_ROOT\n    / "results"\n    / "05"\n    / "best_xgboost_model.joblib"\n)\n\n\n@st.cache_resource\ndef load_model_bundle():\n    if not MODEL_PATH.exists():\n        raise FileNotFoundError(\n            f"Cannot find the model file: {MODEL_PATH}"\n        )\n\n    return joblib.load(MODEL_PATH)\n\n\ntry:\n    model_bundle = load_model_bundle()\nexcept Exception as error:\n    st.error(f"Could not load the Week 7 model. {error}")\n    st.stop()\n\n\npreprocessor = model_bundle["preprocessor"]\nmodel = model_bundle["model"]\nnumeric_features = model_bundle["numeric_features"]\ncategorical_features = model_bundle["categorical_features"]\n\n\n# -----------------------------\n# User inputs\n# -----------------------------\nliving_area = st.number_input(\n    "Living Area (square feet)",\n    min_value=0.0,\n    value=1800.0,\n    step=100.0,\n)\n\nbeds = st.number_input(\n    "Beds",\n    min_value=0,\n    value=3,\n    step=1,\n)\n\nbaths = st.number_input(\n    "Baths",\n    min_value=0,\n    value=2,\n    step=1,\n)\n\nlot_size = st.number_input(\n    "Lot Size (square feet)",\n    min_value=0.0,\n    value=5000.0,\n    step=100.0,\n)\n\n\n# -----------------------------\n# Build one model input row\n# -----------------------------\ndef make_input_data(\n    living_area_value,\n    beds_value,\n    baths_value,\n    lot_size_value,\n):\n    # Start all numeric features as missing.\n    # The saved Week 7 preprocessor will fill them\n    # using the medians learned from training data.\n    input_row = {\n        feature: np.nan\n        for feature in numeric_features\n    }\n\n    # Use "Unknown" for category features that are\n    # not entered in this simple Week 9 app.\n    for feature in categorical_features:\n        input_row[feature] = "Unknown"\n\n    # Add the four user inputs to the matching\n    # Week 7 model features.\n    bedroom_names = [\n        "BedroomsTotal",\n        "Bedrooms",\n        "BedsTotal",\n        "Beds",\n        "bedrooms",\n    ]\n\n    bathroom_names = [\n        "BathroomsTotalInteger",\n        "BathroomsTotalDecimal",\n        "BathroomsFull",\n        "Bathrooms",\n        "BathsTotal",\n        "Baths",\n        "bathrooms",\n    ]\n\n    for name in bedroom_names:\n        if name in input_row:\n            input_row[name] = float(beds_value)\n\n    for name in bathroom_names:\n        if name in input_row:\n            input_row[name] = float(baths_value)\n\n    if "LivingArea" in input_row:\n        input_row["LivingArea"] = float(living_area_value)\n\n    # The Week 7 data contains both lot size columns.\n    # Both use the user\'s Week 9 lot-size input here.\n    if "LotSizeSquareFeet" in input_row:\n        input_row["LotSizeSquareFeet"] = float(lot_size_value)\n\n    if "LotSizeArea" in input_row:\n        input_row["LotSizeArea"] = float(lot_size_value)\n\n    # Week 6 created this feature, so calculate it\n    # from the user-entered beds and baths.\n    if "bed_bath_ratio" in input_row:\n        if baths_value > 0:\n            input_row["bed_bath_ratio"] = (\n                float(beds_value) / float(baths_value)\n            )\n        else:\n            input_row["bed_bath_ratio"] = np.nan\n\n    return pd.DataFrame(\n        [input_row],\n        columns=numeric_features + categorical_features,\n    )\n\n\n# -----------------------------\n# Make the prediction\n# -----------------------------\nif st.button("Predict Price"):\n    input_data = make_input_data(\n        living_area,\n        beds,\n        baths,\n        lot_size,\n    )\n\n    try:\n        prepared_data = preprocessor.transform(input_data)\n        predicted_price = float(model.predict(prepared_data)[0])\n\n        st.success(\n            f"Predicted Price: ${predicted_price:,.0f}"\n        )\n\n        st.caption(\n            "This is a simple Week 9 demo. "\n            "The Week 7 model uses more features than the four inputs "\n            "shown here. Missing numeric features are filled by the "\n            "saved preprocessing step, and unavailable category "\n            "information is set to Unknown."\n        )\n\n    except Exception as error:\n        st.error(f"Prediction failed. {error}")\n'

app_path.write_text(
    app_code,
    encoding="utf-8",
)

print("Created:", app_path)

Created: C:\Users\18327\IDX Exchange\results\09\app.py


## 6. Check the generated Python file

This does not start Streamlit. It only checks that `app.py` contains valid Python syntax.

In [6]:
app_source = app_path.read_text(encoding="utf-8")

compile(
    app_source,
    str(app_path),
    "exec",
)

print("app.py syntax check passed.")
print("Number of lines:", len(app_source.splitlines()))

app.py syntax check passed.
Number of lines: 199


## 7. Test one prediction inside the notebook

This quick test uses example values to make sure the saved preprocessing and model can make a prediction before the Streamlit app is started.

Example:

Living Area = 1,800 square feet

Beds = 3

Baths = 2

Lot Size = 5,000 square feet


In [7]:
def make_test_input(
    living_area_value,
    beds_value,
    baths_value,
    lot_size_value,
):
    input_row = {
        feature: np.nan
        for feature in numeric_features
    }

    for feature in categorical_features:
        input_row[feature] = "Unknown"

    bedroom_names = [
        "BedroomsTotal",
        "Bedrooms",
        "BedsTotal",
        "Beds",
        "bedrooms",
    ]

    bathroom_names = [
        "BathroomsTotalInteger",
        "BathroomsTotalDecimal",
        "BathroomsFull",
        "Bathrooms",
        "BathsTotal",
        "Baths",
        "bathrooms",
    ]

    for name in bedroom_names:
        if name in input_row:
            input_row[name] = float(beds_value)

    for name in bathroom_names:
        if name in input_row:
            input_row[name] = float(baths_value)

    if "LivingArea" in input_row:
        input_row["LivingArea"] = float(living_area_value)

    if "LotSizeSquareFeet" in input_row:
        input_row["LotSizeSquareFeet"] = float(lot_size_value)

    if "LotSizeArea" in input_row:
        input_row["LotSizeArea"] = float(lot_size_value)

    if "bed_bath_ratio" in input_row:
        if baths_value > 0:
            input_row["bed_bath_ratio"] = (
                float(beds_value) / float(baths_value)
            )
        else:
            input_row["bed_bath_ratio"] = np.nan

    return pd.DataFrame(
        [input_row],
        columns=numeric_features + categorical_features,
    )


test_input = make_test_input(
    living_area_value=1800,
    beds_value=3,
    baths_value=2,
    lot_size_value=5000,
)

test_ready = preprocessor.transform(test_input)
test_prediction = float(model.predict(test_ready)[0])

print(f"Example predicted price: ${test_prediction:,.0f}")

Example predicted price: $503,349


## 8. Install Streamlit if needed

Run this cell once. If Streamlit is already installed, nothing will change.

In [8]:
if importlib.util.find_spec("streamlit") is None:
    print("Streamlit is not installed. Installing it now...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "streamlit"]
    )
    print("Streamlit installation finished.")
else:
    print("Streamlit is already installed.")

Streamlit is already installed.


## 9. Run the Streamlit app

After all cells finish, open a Terminal in Jupyter.

Make sure the Terminal is in the main `IDX Exchange` project folder.

Then run:

`streamlit run results/09/app.py`

Streamlit will give you a local web address. Open that address in your browser.

The app will show four inputs:

1. Living Area
2. Beds
3. Baths
4. Lot Size

Click **Predict Price** to see the predicted home price.


## 10. Final Summary

The Week 9 task is complete when `results/09/app.py` has been created and the Streamlit page opens successfully.

The required deliverable is:

`results/09/app.py`

The app loads the trained Week 7 XGBoost model using `joblib`, accepts the four requested property inputs, and returns a predicted price.

This is a simple demonstration app. Because the Week 7 model was trained with additional property and location features, the saved preprocessing step fills unavailable numeric information using its training medians, while unavailable categorical information is set to `Unknown`.


In [9]:
print("Week 9 notebook is complete.")
print("Required deliverable:", app_path)
print()
print("Run the app with:")
print("streamlit run results/09/app.py")

Week 9 notebook is complete.
Required deliverable: C:\Users\18327\IDX Exchange\results\09\app.py

Run the app with:
streamlit run results/09/app.py
